# langnostic — Colab runner (T4)

All model work happens here: dataset streaming, whisper transcription, LoRA training.
Only `results/cache/*.jsonl` (+ `.meta.json`) goes back to the laptop.

Runtime → Change runtime type → **T4 GPU**.

In [ ]:
!nvidia-smi
import torch
print(torch.__version__, torch.cuda.is_available())

In [ ]:
# fresh session: clone; existing repo: pull
import os
REPO = 'https://github.com/Coding-With-Josh/langnostic.git'
if not os.path.isdir('/content/langnostic'):
    !git clone $REPO /content/langnostic
%cd /content/langnostic
!git pull --ff-only

In [ ]:
!pip -q install -r requirements.txt

In [ ]:
# Build speaker-disjoint manifests. One NaijaVoices shard (~4.7 GB) is enough for
# 2-3k rows; VoxPopuli accent rows are evaluation-only.
!PYTHONPATH=. python -m speechgap.data build --out-dir data \
    --naija yoruba:0-0 \
    --voxpopuli en_accented:test en:test \
    --accents en_nl,en_de,en_cs
!PYTHONPATH=. python -m speechgap.data verify --out-dir data

In [ ]:
import json
audit = json.load(open('data/audit.json'))
print(json.dumps({k: audit[k] for k in ('splits', 'groups', 'dropped', 'accents_seen', 'problems')}, indent=2))

In [ ]:
# Zero-shot baseline. Deterministic: pinned model revision, forced per-row language.
!PYTHONPATH=. python -m speechgap.eval --manifest data/manifest_test.jsonl \
    --device cuda --out results/cache/zero_shot.jsonl

# After a disconnect, rerun the same command with --resume.

In [ ]:
!PYTHONPATH=. python -m speechgap.report results/cache/zero_shot.jsonl \
    --out results/zero_shot_report.json

## LoRA adapters (15 and 30 minutes)

Trains only on `manifest_train.jsonl`, early-stops on `manifest_val.jsonl`, and
re-checks that no train/val speaker appears in `manifest_test.jsonl` before it starts.

In [ ]:
for minutes in (15, 30):
    !PYTHONPATH=. python -m speechgap.train --train data/manifest_train.jsonl \
        --val data/manifest_val.jsonl --minutes $minutes \
        --group yoruba_read --language yo --out checkpoints/yo-${minutes}min

In [ ]:
!PYTHONPATH=. python -m speechgap.eval --manifest data/manifest_test.jsonl \
    --device cuda --adapter checkpoints/yo-15min --out results/cache/yo_lora15.jsonl
!PYTHONPATH=. python -m speechgap.eval --manifest data/manifest_test.jsonl \
    --device cuda --adapter checkpoints/yo-30min --out results/cache/yo_lora30.jsonl

for name in ('zero_shot', 'yo_lora15', 'yo_lora30'):
    !PYTHONPATH=. python -m speechgap.report results/cache/$name.jsonl --out results/${name}_report.json

In [ ]:
# Bring the cache home. Transcripts only: no audio, no manifests, no weights.
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/langnostic/cache
!cp results/cache/*.jsonl results/cache/*.meta.json /content/drive/MyDrive/langnostic/cache/

## next

LoRA training is the remaining piece. It reads `data/manifest_train.jsonl` and
`data/manifest_val.jsonl`, trains speaker-disjoint from test, and writes adapters to
`checkpoints/` (never committed). Then evaluate every adapter on every group.